In [18]:
import pandas as pd
import numpy as np
import os
import chess
import lightgbm as lgb
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.decomposition import PCA
from tqdm import tqdm
import itertools

def load_data(csv_path, embeddings_path, num_rows=None):
    df = pd.read_csv(csv_path)
    if num_rows:
        df = df.head(num_rows)
    maia_embeddings = np.load(embeddings_path)
    if len(maia_embeddings) != len(df):
        maia_embeddings = maia_embeddings[:len(df)]
    return df, maia_embeddings

def extract_board_stats(fen):
    board = chess.Board(fen)
    counts = {
        'white_pieces': int(board.occupied_co[chess.WHITE]).bit_count(),
        'black_pieces': int(board.occupied_co[chess.BLACK]).bit_count(),
        'material_balance': sum([len(board.pieces(piece, chess.WHITE)) * val 
                               for piece, val in zip([1,2,3,4,5,6], [1,3,3,5,9,0])]) - \
                           sum([len(board.pieces(piece, chess.BLACK)) * val 
                               for piece, val in zip([1,2,3,4,5,6], [1,3,3,5,9,0])])
    }
    return counts

def build_features(df):
    feats = pd.DataFrame(index=df.index)
    feats['SolutionLength'] = df['Moves'].apply(lambda x: len(str(x).split()))
    feats['IsWhiteToMove'] = df['FEN'].apply(lambda x: 1 if x.split()[1] == 'w' else 0)
    stats = df['FEN'].apply(extract_board_stats).apply(pd.Series)
    feats = pd.concat([feats, stats], axis=1)
    prob_cols = [c for c in df.columns if 'success_prob_blitz' in c]
    feats = pd.concat([feats, df[prob_cols]], axis=1)
    return feats

def encode_themes(df):
    themes_list = df['Themes'].apply(lambda x: x.split())
    mlb = MultiLabelBinarizer()
    themes_encoded = mlb.fit_transform(themes_list)
    themes_df = pd.DataFrame(themes_encoded, columns=mlb.classes_, index=df.index)
    return themes_df

In [16]:
csv_path = "./data/sampple5.csv"
embeddings_path = "./features/maia1_embeddings_sequence5.npy"

df, maia_4d = load_data(csv_path, embeddings_path)
X_struct = build_features(df)
X_themes = encode_themes(df)
X_maia = np.mean(maia_4d, axis=1).reshape(len(maia_4d), -1)
X = np.hstack([X_struct.values, X_themes.values, X_maia])
# X = np.hstack([X_struct.values, X_themes.values])
# X = np.hstack([X_themes.values])
y = df['Rating'].values
kf = KFold(n_splits=5, shuffle=True, random_state=42)
mses = []
results = []

In [21]:
X_maia_flat = maia_4d.reshape(len(maia_4d), -1)
n_components = 800  # tune this
pca = PCA(n_components=n_components)
X_maia_pca = pca.fit_transform(X_maia_flat)
print(f"Original shape: {X_maia_flat.shape}")
print(f"PCA shape: {X_maia_pca.shape}")
print(f"Explained variance: {pca.explained_variance_ratio_.sum():.2%}")

Original shape: (50000, 61440)
PCA shape: (50000, 500)
Explained variance: 73.03%


In [15]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1, random_state=42)
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'verbosity': -1,
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'device': 'cpu'
}
train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)
model = lgb.train(
    params,
    train_data,
    num_boost_round=1000,
    valid_sets=[train_data, val_data],
    valid_names=['train', 'valid'],
    callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(period=100)]
)
preds = model.predict(X_val)
mse = mean_squared_error(y_val, preds)
rmse = np.sqrt(mse)
print(f"Validation RMSE: {rmse:.2f}")
result = pd.DataFrame([{'MSE': mse, 'RMSE': rmse}])
result.to_csv("./maia1EmbedingSequence.csv", index=False)

Training until validation scores don't improve for 50 rounds
[100]	train's rmse: 288.933	valid's rmse: 305.964
[200]	train's rmse: 262.283	valid's rmse: 299.394
[300]	train's rmse: 241.663	valid's rmse: 297.474
[400]	train's rmse: 224.095	valid's rmse: 296.932
[500]	train's rmse: 208.472	valid's rmse: 296.774
Early stopping, best iteration is:
[485]	train's rmse: 210.656	valid's rmse: 296.656
Validation RMSE: 296.66
